## Training Data Preparation

Training data needs to be stored as image, outline mask, and vein mask in one folder.

Images are currently all stored in one folder, outline mask and vein mask need to be read from Label-Studio annotations JSON.  
Images and masks should also be resized to 448x448 px.

In [167]:
import json
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
import random

In [5]:
root_path = "/masc_shared/ag_nauss/projects/phytoakmeter/IEE/vein_segmentation_training_data"

annotations_filename = "photo_annotations_outline_midrib_secveins.json"
path = os.path.join(root_path, annotations_filename)
with open(path, "r") as f:
    annotations = json.load(f)

In [108]:
# sample path

annotation = annotations[1]

In [7]:
annotation["file_upload"]

'CEE_DE_1M_P25070601000110_135_136_20A_M_1_01_leaf.jpg'

In [107]:
annotation

{'id': 263,
 'annotations': [{'id': 16,
   'completed_by': 1,
   'result': [{'original_width': 1011,
     'original_height': 702,
     'image_rotation': 0,
     'value': {'points': [[90.1194288361227, 41.14084223141079],
       [73.58870561698185, 44.273351073544426],
       [56.73171812377903, 46.93598358935802],
       [43.35488288723742, 50.225117873598336],
       [32.843936922240346, 54.76725569469211],
       [27.188689505165847, 56.96001188418567],
       [10.331702011963023, 59.77926984210594],
       [27.623708537248504, 56.490135557865614],
       [33.278955954323, 54.45400481047875],
       [43.93692224034801, 49.755241547278295],
       [57.42251223491027, 46.62273270514466],
       [74.00761283306144, 43.6468493051177],
       [89.015769439913, 40.827591347197426]],
      'closed': True,
      'polygonlabels': ['midrib_polygon']},
     'id': 'orFEBQDJ_E',
     'from_name': 'midrib_polygon',
     'to_name': 'image',
     'type': 'polygonlabels',
     'origin': 'manual'},
  

In [163]:
def resized_outline_mask(

    data: dict,
    dst_path: str|None = None,
    size: tuple[int, int] = (448, 448),
    
) -> NDArray:

    try:
        
        ann = data['annotations'][0]
        outline = next(r for r in ann['result'] if r['value'].get('polygonlabels') == ['outline'])
        
        w = outline['original_width']
        h = outline['original_height']
        pts_pct = outline['value']['points']
        
        # convert percentage coords to pixel coords
        pts = np.array([[p[0] / 100 * w, p[1] / 100 * h] for p in pts_pct], dtype=np.int32)
        
        mask = np.full((h, w), 255, dtype=np.uint8)  # white background
        cv2.fillPoly(mask, [pts], color=0)           # black foreground
        resized = cv2.resize(mask, size, interpolation=cv2.INTER_NEAREST)

        if dst_path is not None:
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            cv2.imwrite(dst_path, resized)
        
        return resized
        
    except Exception as e:
        
        print(e)

In [164]:
def resized_vein_mask(

    data: dict,
    outline_mask: NDArray,
    dst_path: str|None = None,
    size: tuple[int, int] = (448, 448),

) -> NDArray:

    try:
        ann = data['annotations'][0]
        veins = [r for r in ann['result']
                 if r['value'].get('polygonlabels') in (['midrib_polygon'], ['sec_vein_polygon'])]
        w = veins[0]['original_width']
        h = veins[0]['original_height']
        
        vein_mask = np.full((h, w), 255, dtype=np.uint8)  # white background
        
        for v in veins:
            pts_pct = v['value']['points']
            pts = np.array([[p[0] / 100 * w, p[1] / 100 * h] for p in pts_pct], dtype=np.int32)
            cv2.fillPoly(vein_mask, [pts], color=0)  # black foreground

        resized_vein_mask = cv2.resize(vein_mask, size, interpolation=cv2.INTER_NEAREST)
        assert resized_vein_mask.shape == outline_mask.shape
        # clip to the outline mask to exclude midribs that were drawn "through" other leaves obscuring the target leaf
        resized_vein_mask[outline_mask == 255] = 255
    
        if dst_path is not None:
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            cv2.imwrite(dst_path, resized_vein_mask)
        
        return resized_vein_mask

    except Exception as e:

        print(e)

In [165]:
def resized_masked_image(
    
    src_path: str,
    mask: NDArray,
    dst_path: str|None = None,
    size: tuple[int, int] = (448, 448),

) -> NDArray:

    try:
        
        src = cv2.imread(src_path)
        resized = cv2.resize(src, (448, 448), interpolation=cv2.INTER_LINEAR)
        keep_mask = cv2.bitwise_not(mask) # outline_mask is inverted, so inverted back here
        masked = cv2.bitwise_and(resized, resized, mask=keep_mask)
        masked[keep_mask == 0] = 255 # return to inverted to match outline_mask

        if dst_path is not None:
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            cv2.imwrite(dst_path, masked)
        
        return masked
        
    except Exception as e:
        
        print(e)

In [182]:
k_pretrain, k_val = 15, 5 # 25 annotations, 15 used for pretraining, 5 for validation, 5 for testing

idx_all = list(range(len(annotations)))

idx_pretrain = random.sample(idx_all, k_pretrain)
remaining = [i for i in idx_all if i not in idx_pretrain]

idx_val = random.sample(remaining, k_val)
idx_test = [i for i in remaining if i not in idx_val]

partitions = {
    "partition": ["pretrain", "valid", "test"],
    "indices":[idx_pretrain, idx_val, idx_test]
   }

In [195]:
dst_size = (448, 448)
dataset = "photo"

for partition, idx in zip(partitions["partition"], partitions["indices"]):

    partition_path = os.path.join(root_path, dataset+"_labels", partition)
    partition_annotations = [annotations[i] for i in idx]

    for annotation in partition_annotations:
    
        leaf_id = "_".join(os.path.splitext(annotation["file_upload"])[0].split("_")[:-1])
        leaf_dir = os.path.join(partition_path, leaf_id)
    
        outline_dst_path = os.path.join(leaf_dir, "outline.png")
        outline_mask = resized_outline_mask(annotation, dst_path=outline_dst_path, size=dst_size)
    
        vein_dst_path = os.path.join(leaf_dir, "vein.png")
        resized_vein_mask(annotation, outline_mask, dst_path=vein_dst_path, size=dst_size)
        
        img_src_path = os.path.join(root_path, "photo", annotation["file_upload"])
        img_dst_path = os.path.join(leaf_dir, "leaf.png")
        resized_masked_image(img_src_path, outline_mask, dst_path=img_dst_path, size=dst_size)